# 19.9 接上較快的做法，先檢查什麼？


助理每生成一個新字，都需要知道前文。若每次把整段對話重新算一遍，會重複處理已讀過的位置；KV cache 保存各層已算好的 Key、Value，讓下一步沿用它們。但省下重算以前，必須確認前文、位置與可讀範圍仍相同，否則兩條路可能已在回答不同問題。

本章實際交付的核心採用下列設定。這些做法各自改變一部分計算，不能合成一句「最佳化後品質與速度都更好」。

| 實際選用的做法 | 在這份核心中做什麼 |
| --- | --- |
| RoPE | 旋轉 Query、Key 的特徵，讓注意力使用位置資訊。 |
| RMSNorm | 按特徵的均方根調整尺度，再使用可學的縮放係數。 |
| GQA | 4 個 Query 頭共用 2 組 Key、Value，減少這部分的權重與快取。 |
| tied embeddings | 查字與產生下一字分數共用同一張權重表。 |
| SDPA | 用 PyTorch 的縮放點積注意力介面計算注意力。 |
| top-2 MoE | 每層有 4 個 expert，每個有效 token 選其中 2 個前饋網路。 |

前幾項的機制見[第14章](https://birdhackor.github.io/tiny-perceptron-vlm/chapter-14.html)與[第16章](https://birdhackor.github.io/tiny-perceptron-vlm/chapter-16.html)，MoE 的路由見[第15章](https://birdhackor.github.io/tiny-perceptron-vlm/chapter-15.html)。**SDPA 是介面，FlashAttention 是可能的底層運算實作**；裝置、數值格式與形狀都會影響選擇。本成品的設定記錄支持「使用 SDPA」，沒有支持「已選到 FlashAttention 核心」。同樣，Python 逐 expert 分派不等於專用稀疏運算核心。實際設定可查[最終模型記錄](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/selftrained/results/v2-final-public-results.json)，運算路徑可查[模型程式](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/tiny_perceptron/selftrained/model.py)。

先用同一套實作的**局部微型示範**核對快取：模型縮成寬16、1層，仍用隨機權重；5個人工 ID 只是機制輸入，不是一道已學會的問答。一路一次讀完，另一路先讀3格、再接2格，比較最後位置的32項候選分數。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.selftrained.model import LimitedAssistant, SelftrainedConfig

torch.manual_seed(42)
config = SelftrainedConfig(vocab_size=32, width=16, layers=1, heads=2, kv_heads=1, ffn_hidden=32, top_k=2)
core = LimitedAssistant(config).eval()
ids = torch.tensor([[1, 11, 12, 13, 14]])
split = 3
with torch.no_grad():
    full = core(ids)["logits"][:, -1]
    cache = core(ids[:, :split])["cache"]
    cached = core(ids[:, split:], cache=cache)["logits"][:, -1]
print("最大分數差", (full - cached).abs().max().item())
print("同一計算", torch.allclose(full, cached, atol=1e-4, rtol=1e-4))

在本次 CPU 核對中，最大差約為 `4.8e-7`，容差比較為 True；這支持本例的兩路計算接近。`eval()`停用訓練模式中的隨機行為，`no_grad()`省去求梯度；程式沒有更新權重，也沒有量速度。

公開成品的生成預設已使用 KV cache。圖片與聲音在首次計算時編成特徵，放進完整前文；後續生成沿用快取，不重新編碼素材。另一項常見做法是[sequence packing](https://birdhackor.github.io/tiny-perceptron-vlm/16.5.html)：把短紀錄裝到同一序列，再用遮罩隔開。這份正式訓練仍是一筆紀錄佔一列，較短列補 PAD，沒有採用 packing；前章的局部示範不能當成它已套用的證據。

練習只把 `split=3` 改成2，先預測兩路仍讀相同5個位置，最後分數應接近，再執行核對。若要宣稱某條路更快，下一步才是在同一硬體、同一權重與生成條件下量時間和記憶體；快取的一致性與模型的回答品質要分開評估。
